@inproceedings{perera-etal-2025-indonlp,
    title = "{I}ndo{NLP} 2025 Shared Task: {R}omanized {S}inhala to {S}inhala Reverse Transliteration Using {BERT}",
    author = "Perera, Sandun Sameera  and
      Jayakodi, Lahiru Prabhath  and
      Sumanathilaka, Deshan Koshala  and
      Anuradha, Isuri",
    editor = "Weerasinghe, Ruvan  and
      Anuradha, Isuri  and
      Sumanathilaka, Deshan",
    booktitle = "Proceedings of the First Workshop on Natural Language Processing for Indo-Aryan and Dravidian Languages",
    month = jan,
    year = "2025",
    address = "Abu Dhabi",
    publisher = "Association for Computational Linguistics",
    url = "https://aclanthology.org/2025.indonlp-1.16/",
    pages = "135--140",
    abstract = "The Romanized text has become popu lar with the growth of digital communi cation platforms, largely due to the fa miliarity with English keyboards. In Sri Lanka, Romanized Sinhala, commonly re ferred to as {\textquotedblleft}Singlish{\textquotedblright} is widely used in digi tal communications. This paper introduces a novel context-aware back-transliteration system designed to address the ad-hoc typ ing patterns and lexical ambiguity inher ent in Singlish. The proposed system com bines dictionary-based mapping for Singlish words, a rule-based transliteration for out of-vocabulary words and a BERT-based language model for addressing lexical am biguities. Evaluation results demonstrate the robustness of the proposed approach, achieving high BLEU scores along with low Word Error Rate (WER) and Character Er ror Rate (CER) across test datasets. This study provides an effective solution for Ro manized Sinhala back-transliteration and establishes the foundation for improving NLP tools for similar low-resourced lan guages."
}

In [ ]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 26.9 MB/s eta 0:00:00


In [ ]:
import itertools
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMaskedLM
import time
from itertools import groupby
import re
from functools import reduce
from collections import defaultdict
from jiwer import wer, cer
from nltk.translate.bleu_score import corpus_bleu

In [ ]:

nVowels =26
# 46
consonants = []
consonantsUni = []

# 26
vowels = []
vowelsUni = []
vowelModifiersUni = []

# 6
specialConsonants = []
specialConsonantsUni = []
# 2
specialCharUni = []
specialChar = []


def initializeVar():

    vowelsUni.append("ඌ")
    vowels.append("oo")
    vowelModifiersUni.append("ූ")

    vowelsUni.append("ඕ")
    vowels.append("o\\)")
    vowelModifiersUni.append("ෝ")
    vowelsUni.append("ඕ")
    vowels.append("oe")
    vowelModifiersUni.append("ෝ")
    vowelsUni.append("ආ")
    vowels.append("aa")
    vowelModifiersUni.append("ා")
    vowelsUni.append("ආ")
    vowels.append("a\\)")
    vowelModifiersUni.append("ා")
    vowelsUni.append("ඈ")
    vowels.append("Aa")
    vowelModifiersUni.append("ෑ")
    vowelsUni.append("ඈ")
    vowels.append("A\\)")
    vowelModifiersUni.append("ෑ")
    vowelsUni.append("ඈ")
    vowels.append("ae")
    vowelModifiersUni.append("ෑ")
    vowelsUni.append("ඊ")
    vowels.append("ii")
    vowelModifiersUni.append("ී")
    vowelsUni.append("ඊ")
    vowels.append("i\\)")
    vowelModifiersUni.append("ී")
    vowelsUni.append("ඊ")
    vowels.append("ie")
    vowelModifiersUni.append("ී")
    vowelsUni.append("ඊ")
    vowels.append("ee")
    vowelModifiersUni.append("ී")
    vowelsUni.append("ඒ")
    vowels.append("ea")
    vowelModifiersUni.append("ේ")
    vowelsUni.append("ඒ")
    vowels.append("e\\)")
    vowelModifiersUni.append("ේ")
    vowelsUni.append("ඒ")
    vowels.append("ei")
    vowelModifiersUni.append("ේ")
    vowelsUni.append("ඌ")
    vowels.append("uu")
    vowelModifiersUni.append("ූ")
    vowelsUni.append("ඌ")
    vowels.append("u\\)")
    vowelModifiersUni.append("ූ")

    vowelsUni.append("ඖ")
    vowels.append("au")
    vowelModifiersUni.append("ෞ")

    vowelsUni.append("ඇ")
    vowels.append("\\a")
    vowelModifiersUni.append("ැ")

    vowelsUni.append("අ")
    vowels.append("a")
    vowelModifiersUni.append("")

    vowelsUni.append("ඇ")
    vowels.append("A")
    vowelModifiersUni.append("ැ")
    vowelsUni.append("ඉ")
    vowels.append("i")
    vowelModifiersUni.append("ි")
    vowelsUni.append("එ")
    vowels.append("e")
    vowelModifiersUni.append("ෙ")
    vowelsUni.append("උ")
    vowels.append("u")
    vowelModifiersUni.append("ු")
    vowelsUni.append("ඔ")
    vowels.append("o")
    vowelModifiersUni.append("ො")
    vowelsUni.append("ඓ")
    vowels.append("I")
    vowelModifiersUni.append("ෛ")

    specialConsonantsUni.append("ං")
    specialConsonants.append("\\n")

    specialConsonantsUni.append("ඃ")
    specialConsonants.append("\\h")
    specialConsonantsUni.append("ඞ")
    specialConsonants.append("\\N")
    specialConsonantsUni.append("ඍ")
    specialConsonants.append("\\R")
    # special characher Repaya
    specialConsonantsUni.append("ර්"+"\u200D")
    specialConsonants.append("R")
    specialConsonantsUni.append("ර්"+"\u200D")
    specialConsonants.append("\\r")

    consonantsUni.append("ඬ")
    consonants.append("nnd")

    consonantsUni.append("ඳ")
    consonants.append("nndh")

    consonantsUni.append("ඟ")
    consonants.append("nng")

    consonantsUni.append("ත")
    consonants.append("th")

    consonantsUni.append("ධ")
    consonants.append("dh")
    consonantsUni.append("ඝ")
    consonants.append("gh")
    consonantsUni.append("ච")
    consonants.append("ch")
    consonantsUni.append("ඵ")
    consonants.append("ph")
    consonantsUni.append("භ")
    consonants.append("bh")
    consonantsUni.append("ඣ")
    consonants.append("jh")
    consonantsUni.append("ෂ")
    consonants.append("sh")
    consonantsUni.append("ඥ")
    consonants.append("GN")
    consonantsUni.append("ඤ")
    consonants.append("KN")
    consonantsUni.append("ළු")
    consonants.append("Lu")
    consonantsUni.append("ඛ")
    consonants.append("kh")
    consonantsUni.append("ඨ")
    consonants.append("Th")
    consonantsUni.append("ඪ")
    consonants.append("Dh")
    consonantsUni.append("ශ")
    consonants.append("S")
    consonantsUni.append("ද")
    consonants.append("d")
    consonantsUni.append("ච")
    consonants.append("c")
    consonantsUni.append("ත")
    consonants.append("th")
    consonantsUni.append("ට")
    consonants.append("t")
    consonantsUni.append("ක")
    consonants.append("k")
    consonantsUni.append("ඩ")
    consonants.append("D")
    consonantsUni.append("න")
    consonants.append("n")
    consonantsUni.append("ප")
    consonants.append("p")
    consonantsUni.append("බ")
    consonants.append("b")
    consonantsUni.append("ම")
    consonants.append("m")
    consonantsUni.append("‍ය")
    consonants.append("\\u005C" + "y")
    consonantsUni.append("‍ය")
    consonants.append("Y")
    consonantsUni.append("ය")
    consonants.append("y")
    consonantsUni.append("ජ")
    consonants.append("j")
    consonantsUni.append("ල")
    consonants.append("l")
    consonantsUni.append("ව")
    consonants.append("v")
    consonantsUni.append("ව")
    consonants.append("w")
    consonantsUni.append("ස")
    consonants.append("s")
    consonantsUni.append("හ")
    consonants.append("h")
    consonantsUni.append("ණ")
    consonants.append("N")
    consonantsUni.append("ළ")
    consonants.append("L")
    consonantsUni.append("ඛ")
    consonants.append("K")
    consonantsUni.append("ඝ")
    consonants.append("G")
    consonantsUni.append("ඵ")
    consonants.append("P")
    consonantsUni.append("ඹ")
    consonants.append("B")
    consonantsUni.append("ෆ")
    consonants.append("f")
    consonantsUni.append("ග")
    consonants.append("g")
    # last because we need to ommit this in dealing with Rakaransha
    consonantsUni.append("ර")
    consonants.append("r")
    specialCharUni.append("ෲ")
    specialChar.append("ruu")
    specialCharUni.append("ෘ")
    specialChar.append("ru")
    # specialCharUni[2]="්‍ර" specialChar[2]="ra"

initializeVar()

In [ ]:
def convertText(text):
    s=""
    r=""
    v=""
    # text = document.txtBox.box1.value;
    # special consonents
    for i in range(len(specialConsonants)):
        text = text.replace(specialConsonants[i], specialConsonantsUni[i])
# consonents + special Chars
    for i in range(len(specialCharUni)):
        for j in range(len(consonants)):
            s = consonants[j] + specialChar[i]
            v = consonantsUni[j] + specialCharUni[i]
            # r = new RegExp(s, "g")
            r = s.replace(s+"/G", "")
            text = text.replace(r, v)

# consonants + Rakaransha + vowel modifiers
    for j in range(len(consonants)):
        for i in range(len(vowels)):
            s = consonants[j] + "r" + vowels[i]
            v = consonantsUni[j] + "්‍ර" + vowelModifiersUni[i]
            # r = new RegExp(s, "g")
            r = s.replace(s+"/G", "")
            text = text.replace(r, v)
        s = consonants[j] + "r"
        v = consonantsUni[j] + "්‍ර"
        # r = new RegExp(s, "g")
        r = s.replace(s+"/G", "")
        text = text.replace(r, v)
# consonents + vowel modifiers
    for i in range(len(consonants)):
        for j in range(nVowels):
            s = consonants[i] + vowels[j]
            v = consonantsUni[i] + vowelModifiersUni[j]
            # r = new RegExp(s, "g")
            r = s.replace(s+"/G", "")
            text = text.replace(r, v)

# consonents + HAL
    for i in range(len(consonants)):
        r = consonants[i].replace(consonants[i]+"/G", "")
        text = text.replace(r, consonantsUni[i] + "්")
# vowels
    for i in range(len(vowels)):
        # r = new RegExp(vowels[i], "g")
        r = vowels[i].replace(vowels[i]+"/G", "")
        text = text.replace(r, vowelsUni[i])

    return text

## Helper functions

In [ ]:
# Function to read the dictionary file and return a dictionary of Singlish to Sinhala words
def load_dictionary(file_path):
    singlish_to_sinhala = {}

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            # Split the line into Singlish word and Sinhala words part
            parts = line.split(", Sinhala Words: ")
            if len(parts) == 2:
                singlish_word = parts[0].replace("Word: ", "").strip()
                sinhala_words = eval(parts[1].strip())  # Convert the string representation of list to an actual list
                singlish_to_sinhala[singlish_word] = sinhala_words
    return singlish_to_sinhala

In [ ]:
# Function to get the Sinhala word for a given Singlish word
def get_sinhala_word(singlish_word, dictionary):
    # print(f"get_singlish wor func: {dictionary.get(singlish_word, convertText(str(singlish_word)))}")
    # print(type(dictionary.get(singlish_word, convertText(str(singlish_word)))))
    sinhala_word = dictionary.get(singlish_word, "Word not found")
    if sinhala_word == "Word not found":
      sinhala_word = [convertText(str(singlish_word))]
    return sinhala_word

In [ ]:
# Remove words which are not in the BERT vocabulary
def clean_words(sinhala_words, tokenizer):
    new_sinhala_words = []
    vocab = set(tokenizer.vocab)  # Convert vocab to a set for O(1) lookups

    for words in sinhala_words:
        if len(words) == 1:
            new_sinhala_words.append(words)
            continue
        clean_words = [word for word in words if word in vocab]

        if not clean_words:  # If no words are found in the vocab
            clean_words = [words[0]]
        new_sinhala_words.append(clean_words)

    return new_sinhala_words

In [ ]:
# Get the candidate words and mask the sentence accordingly
def process_sentence(new_sinhala_words):
    sinhala_words_processing = new_sinhala_words.copy()
    candidate_words = []

    # Use list comprehension and handle masking
    masked_sentence = []
    for words in sinhala_words_processing:
        if len(words) > 1:
            candidate_words.append(words)
            masked_sentence.append('[MASK]')
        else:
            masked_sentence.append(' '.join(words))

    # Use 'join' to build the sentence efficiently
    masked_sentence = ' '.join(masked_sentence)
    return masked_sentence, candidate_words

In [ ]:
# Split the input sentence and get the relevant native Sinhala words
def get_sinhala_words(singlish_sentence, singlish_to_sinhala_dict):
    # Separate words by spaces and get corresponding Sinhala words
    singlish_words = singlish_sentence.split()
    sinhala_words = [get_sinhala_word(word, singlish_to_sinhala_dict) for word in singlish_words]
    return sinhala_words

In [ ]:
# generate the sentence combinations with candidate words
def generate_sentences_with_all_combinations(template, candidates):
    # Split the template by spaces
    template_parts = template.split()

    # Identify the indices of blanks
    blank_indices = [i for i, word in enumerate(template_parts) if word == "[MASK]"]

    # Generate all combinations of candidate words for blanks
    all_combinations = list(itertools.product(*candidates))

    # Generate all full sentences
    all_sentences = []
    for combo in all_combinations:
        filled_sentence = template_parts.copy()  # Start with the template
        for i, word in zip(blank_indices, combo):  # Replace blanks with the combo words
            filled_sentence[i] = word
        all_sentences.append(" ".join(filled_sentence))  # Join words into a sentence

    return all_sentences

In [ ]:
def fill_blanks(blanked_sentence, combination):
  # Split the sentence by the '[BLANK]' placeholder
  parts = blanked_sentence.split('[MASK]')
  # Rebuild the sentence by joining the parts and filling in the blanks
  filled_sentence = ''.join(f"{part}{word}" for part, word in zip(parts, combination + ('',)))
  return filled_sentence

In [ ]:
def generate_probs(sentence_with_blank, candidates, tokenizer):
  # Tokenize the sentence
  input_ids = tokenizer.encode(sentence_with_blank, return_tensors='pt')

  # Find the position of the mask token
  mask_token_index = torch.where(input_ids == tokenizer.mask_token_id)[1].item()

  # Get the logits (unnormalized probabilities) for each token in the sentence
  with torch.no_grad():
      outputs = model(input_ids)
      logits = outputs.logits

  # Get the logits for the mask token position
  mask_token_logits = logits[0, mask_token_index, :]

  # Get the probabilities for the candidate words
  word_ids = tokenizer.convert_tokens_to_ids(candidates)
  word_probs = F.softmax(mask_token_logits, dim=-1)[word_ids].tolist()

  zipped = zip(candidates, word_probs)
  return zipped

In [ ]:
# generate the sentence combinations with candidate words
def generate_sentences_with_all_combinations(template, candidates):
    # Split the template by spaces
    template_parts = template.split()

    # Identify the indices of blanks
    blank_indices = [i for i, word in enumerate(template_parts) if word == "[MASK]"]

    # Generate all combinations of candidate words for blanks
    all_combinations = list(itertools.product(*candidates))

    # Generate all full sentences
    all_sentences = []
    for combo in all_combinations:
        filled_sentence = template_parts.copy()  # Start with the template
        for i, word in zip(blank_indices, combo):  # Replace blanks with the combo words
            filled_sentence[i] = word
        all_sentences.append(" ".join(filled_sentence))  # Join words into a sentence

    return all_sentences

In [ ]:
# Function to calculate the product of values for a given sentence
def calculate_product(sentence, word_probabilities):
    product = 1
    found = False
    for key, value in word_probabilities.items():
        parts = key.split("--")

        if re.sub(r'\s+', ' ', parts[2]).strip() == sentence:
            product *= value
            found = True
    return product if found else None

In [ ]:
# Generate sentences with one blank
def generate_sentences_with_one_blank(word_combinations, mask_indexes, masked_sentence):
    one_blank_sentences = {}
    for comb in word_combinations:
      for index, mask_index in enumerate(mask_indexes):
        combination = list(comb)
        masking_word = combination[index]
        combination[index] = '[MASK]'
        one_blank_sentence = fill_blanks(masked_sentence, tuple(combination))

        if one_blank_sentence in one_blank_sentences.keys():
          one_blank_sentences[one_blank_sentence].append(masking_word)
        else:
          one_blank_sentences[one_blank_sentence] = [masking_word]

    return one_blank_sentences

In [ ]:
# Generate probability dictinary
def generate_probability_dict(one_blank_sentences, tokenizer):
    word_probabilities = {}
    for one_blank_sentence, candidate in one_blank_sentences.items():
      probs = list(generate_probs(one_blank_sentence, candidate, tokenizer))

      for i in range(len(probs)):
        word = probs[i][0]
        full_sentence = one_blank_sentence.replace('[MASK]', word)
        sentence_key = one_blank_sentence + "--" + word + "--" + full_sentence
        prob = probs[i][1]
        word_probabilities[sentence_key] = prob

    return word_probabilities

In [ ]:
def calculate_number_of_bert_calls(candidate_words):
    n = len(candidate_words)
    # If there's only one blank, there is only one possible sentence with that blank remaining
    if n == 1:
        return 1

    total_sentences = 0

    # Iterate over each blank
    for i in range(n):
        # Calculate the product of candidate words for all other blanks
        product = reduce(lambda x, y: x * y,
                         (len(candidate_words[j]) for j in range(n) if j != i))
        total_sentences += product

    return total_sentences

In [ ]:
def get_key_by_value(dictionary, value):
    """
    This function searches a dictionary for a given value and returns the corresponding key.
    If the value is not found, it returns None.
    """
    for key, val in dictionary.items():
        if val == value:
            return key
    return None

In [ ]:
def numbering_masks_sentences(sentences):
    # Initialize the mask counter and a dictionary to keep track of mappings
    mask_counter = 1
    mask_map = {}
    previous_masks = []

    updated_sentences = []

    for i, sentence in enumerate(sentences):
        parts = sentence.split("[MASK]")
        labeled_sentence = parts[0]

        for j in range(1, len(parts)):
            # For the first two masks in a sentence (if not the first sentence), use previous masks
            if i > 0 and j <= 2:
                label = previous_masks[j - 1]
            else:
                # Otherwise, assign a new label if not already assigned
                mask_key = (i, j)
                if mask_key in mask_map:
                    label = mask_map[mask_key]
                else:
                    label = f"MASK{mask_counter}"
                    mask_map[mask_key] = label
                    mask_counter += 1

            labeled_sentence += f"[{label}]" + parts[j]

            # Save the last two labels to use in the next sentence for overlap
            if j >= len(parts) - 2:
                previous_masks.append(label)

        # Only retain the last two labels for overlap with the next sentence
        previous_masks = previous_masks[-2:]

        updated_sentences.append(labeled_sentence)

    return updated_sentences

In [ ]:
# Replace [MASK] with numbered labels
def numbering_masks_sentence(sentence):
    masks = re.findall(r'\[MASK\]', sentence)
    for i, _ in enumerate(masks, 1):
        sentence = sentence.replace("[MASK]", f"[MASK{i}]", 1)
    return sentence

In [ ]:
# Function to extract words for each mask
def find_mask_words(mask_sentences, labeled_sentences):
    mask_words = defaultdict(list)

    # Iterate over sentences with masks and corresponding labeled sentences
    for mask_sentence, labeled_sentence in zip(mask_sentences, labeled_sentences):
        mask_parts = re.findall(r'\[MASK\d+\]', mask_sentence)
        labeled_parts = labeled_sentence.split()

        # Iterate over mask parts and map them to corresponding words
        mask_index = 0
        for i, part in enumerate(mask_sentence.split()):
            if mask_index < len(mask_parts) and part == mask_parts[mask_index]:
                word = labeled_parts[i]
                mask_words[mask_parts[mask_index]].append(word)
                mask_index += 1

    # Remove duplicates and format the output
    mask_words = {mask: list(set(words)) for mask, words in mask_words.items()}
    return mask_words

In [ ]:
# Function to replace MASKs and collect candidates
def replace_masks_and_collect_candidates(sentence, mask_words):
    candidates = []

    for mask, words in mask_words.items():
        if len(words) > 1:
            # Replace with [MASK] if there are multiple words
            sentence = sentence.replace(f"[{mask[1:-1]}]", "[MASK]")
            candidates.append(words)
        else:
            # Replace with the single word
            sentence = sentence.replace(f"[{mask[1:-1]}]", words[0])

    return sentence, candidates

In [ ]:
def chunk_sentence(mask_sentence, candidate_words):
  mask_sentence = mask_sentence.split()
  mask_indexes = [index for index, word in enumerate(mask_sentence) if word == '[MASK]']

  # Map candidate_words indexes with mask_indexes
  index_map = {}
  for i in range(len(mask_indexes)):
    index_map[i] = mask_indexes[i]

  max_bert_call = 20
  overlap = 2
  mask_count = 0
  sentences = []
  candidates = []
  sub_sentence = []
  sub_candidates = []


  for index, word in enumerate(mask_sentence):
    if word != "[MASK]":
      sub_sentence.append(word)
      if index == len(mask_sentence)-1:
        sentences.append(" ".join(sub_sentence))
        candidates.append(sub_candidates)
    else:
      sub_candidates.append(candidate_words[get_key_by_value(index_map, index)])
      sub_sentence.append(word)
      mask_count += 1

      if (len(sub_candidates)>= (overlap+1)):

        if (calculate_number_of_bert_calls(sub_candidates)) > max_bert_call:
          try:
            sub_sentence = sub_sentence + mask_sentence[mask_indexes[mask_count-1]+1:mask_indexes[mask_count]]
          except:
            sub_sentence = sub_sentence + mask_sentence[mask_indexes[mask_count-1]+1:]

          sentences.append(" ".join(sub_sentence))
          candidates.append(sub_candidates)

          sub_sentence = mask_sentence[(mask_indexes[(mask_count-overlap)-1])+1:index+1]
          sub_candidates = sub_candidates[-2:]

        elif index == len(mask_sentence)-1:
          sentences.append(" ".join(sub_sentence))
          candidates.append(sub_candidates)

  return sentences, candidates

In [ ]:
# transliterating process
def transliterate(masked_sentence, candidates, tokenizer):

  word_combinations = list(itertools.product(*candidates))
  word_list = masked_sentence.split()
  mask_indexes = [index for index, word in enumerate(word_list) if word == '[MASK]']

  # generate sentences with one blanks including possible candidae words for the blank
  one_blank_sentences = generate_sentences_with_one_blank(word_combinations, mask_indexes, masked_sentence) #dictionary

  word_probabilities = generate_probability_dict(one_blank_sentences, tokenizer)
  full_sentences = generate_sentences_with_all_combinations(masked_sentence, candidates)

  # Find the sentence with the highest product
  max_product = None
  max_sentence = None

  for sentence in full_sentences:
      product = calculate_product(sentence, word_probabilities)
      if product is not None and (max_product is None or product > max_product):
          max_product = product
          max_sentence = sentence

  # print(f"Output: {max_sentence}")
  return max_sentence

In [ ]:
def generate_sinhala(singlish_sentence, singlish_to_sinhala_dict, tokenizer):
  sinhala_words = get_sinhala_words(singlish_sentence, singlish_to_sinhala_dict)
  filtered_sinhala_words = clean_words(sinhala_words, tokenizer)
  # print(f"Filtered words: {filtered_sinhala_words}\n")
  masked_sentence, candidates = process_sentence(filtered_sinhala_words)

  while True:
    if len(candidates) == 0:
      # print(f"Output: {masked_sentence}")
      return masked_sentence
      # return masked_sentence, ["No candidates"]

    else:
      # print(f"Masked sentence: {masked_sentence}")
      # print(f"Candidate words: {candidates}\n")

      if len(candidates) <= 3:
        output = transliterate(masked_sentence, candidates, tokenizer)
        # print(f"Output: {output}")
        return output
      else:
        sentences, candidates = chunk_sentence(masked_sentence, candidates)
        # print(f"CHUNKED Sentences: {sentences}")
        # print(f"CHUNKED Candidates: {candidates}\n")

        # Numbering masks
        numbered_input_sentence = numbering_masks_sentence(masked_sentence)
        # print(f"Numbered input sentence: {numbered_input_sentence}")
        numbered_sentences = numbering_masks_sentences(sentences)
        # print(f"Numbered sentences: {numbered_sentences}")

        filled_sentences = [transliterate(sentences[i], candidates[i], tokenizer) for i in range(len(sentences))]
        # print(f"Filled sentences: {filled_sentences}")

        # Find the words for each mask
        mask_words = find_mask_words(numbered_sentences, filled_sentences)
        # print(f"Mask words: {mask_words}")

        # Replace the MASKs and collect candidates
        masked_sentence, candidates = replace_masks_and_collect_candidates(numbered_input_sentence, mask_words)
        # print(f"Updated sentence: {masked_sentence}")
        # print(f"Updated Candidates: {candidates}\n")

### Model & Tokenizer

In [ ]:
model_name = "Sameera827/sinhala-bert-dakshina_new01"
model_directory = "/content/drive/MyDrive/Singlish-Back-Transliterator/Model"

tokenizer = AutoTokenizer.from_pretrained("Ransaka/sinhala-bert-medium-v2")
model = AutoModelForMaskedLM.from_pretrained(model_name)

# Load the tokenizer and model from the local directory
# tokenizer = AutoTokenizer.from_pretrained(model_directory)
# model = AutoModelForMaskedLM.from_pretrained(model_directory)

config.json:   0%|          | 0.00/640 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/315 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/640 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/202M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

### Load prepared dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

df_full = pd.read_csv('/content/drive/MyDrive/Research Datasets/full_sinhala_dataset.csv')
len(df_full)

638675

In [ ]:
df_full.drop(columns=["Sentiment"], inplace=True)

if df_full['Sentence'].isna().sum() == 0 and df_full['Sentence'].duplicated().sum() == 0:
    print("No missing or duplicate values in Sentence")
else:
    df_full = df_full.dropna(subset=['Sentence'])

No missing or duplicate values in Sentence


### Seperate Singlish rows

In [ ]:
import torch
import pandas as pd

SINHALA_START = 0x0D80
SINHALA_END   = 0x0DFF


def romanized_confidence_filter(
    df,
    text_col="Sentence",
    device="cuda",
    threshold=95.0
):
    """
    Adds a confidence score for romanization and
    keeps rows where confidence >= threshold
    """

    confidences = []
    keep_mask = []

    for text in df[text_col].fillna(""):
        codes = torch.tensor([ord(c) for c in text], device=device)

        if codes.numel() == 0:
            confidences.append(100.0)
            keep_mask.append(True)
            continue

        is_sinhala = (codes >= SINHALA_START) & (codes <= SINHALA_END)
        non_sinhala_ratio = (~is_sinhala).float().mean()

        confidence = (non_sinhala_ratio * 100).item()

        confidences.append(confidence)
        keep_mask.append(confidence >= threshold)

    df = df.copy()
    df["romanized_confidence"] = confidences

    return df[keep_mask].reset_index(drop=True)

In [ ]:
df_romanized = romanized_confidence_filter(
    df_full,
    text_col="Sentence",
    threshold=95.0
)

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
len(df_romanized)

227771

In [ ]:
df_romanized.head(50)

,Sentence,romanized_confidence
0,ubalage magule sim eka gaha ganin .... කේ....🤣🤣🤣,95.833336
1,fb eka mara widiyata slow video ekakwath balan...,96.078438
2,hey friendz මේ app 1n 1999 data package 1activ...,98.726120
3,"yako 3g wath hariyata deeganna baha, kohe 4g d...",96.047440
4,lebsack blaze mini 4g smartphone එක 3495 ta ga...,96.969696
5,♨කලින් gibson quil app ekta register wechha ay...,95.652168
6,uththa 4g පල lankawen thopi signal nathiwa mag...,97.142860
7,99ට 3gb wath dipn nttn bllekwt gnne n,97.297302
8,මෙන්න complaine eka thaama cll ekk.na.mage pho...,95.370369
9,youtube blanna puluwan resolutions eka 360p ද??,97.872337


In [ ]:
df_romanized.to_csv("/content/drive/MyDrive/Singlish-Transliterator/Singlish.csv")

In [ ]:
df_romanized.drop(columns=["romanized_confidence"], inplace=True)

In [ ]:
import pandas as pd

df_romanized = pd.read_csv('/content/drive/MyDrive/Singlish-Transliterator/Singlish.csv')
len(df_romanized)

227771

### Random split

In [ ]:
import numpy as np

df_shuffled = df_romanized.sample(frac=1, random_state=42).reset_index(drop=True)

df_romanized_100000 = df_shuffled.iloc[:100_000]
df_romanized_50000  = df_shuffled.iloc[100_000:150_000]
df_romanized_10000  = df_shuffled.iloc[150_000:160_000]
df_romanized_5000   = df_shuffled.iloc[160_000:165_000]
df_romanized_1000   = df_shuffled.iloc[165_000:166_000]

In [ ]:
print(
    len(df_romanized_100000),
    len(df_romanized_50000),
    len(df_romanized_10000),
    len(df_romanized_5000),
    len(df_romanized_1000)
)

100000 50000 10000 5000 1000


### Load Singlish-Sinhala dictionary

In [ ]:
# Load Dictionary
file_path = "/content/drive/MyDrive/Singlish-Transliterator/SinglishDict/WSD Romanized-Sinhala - Sinhala .txt"
singlish_to_sinhala_dict = load_dictionary(file_path)

### Transliteration

In [ ]:
df_romanized.head()

,Unnamed: 0,Sentence,romanized_confidence
0,0,ubalage magule sim eka gaha ganin .... කේ....🤣🤣🤣,95.833336
1,1,fb eka mara widiyata slow video ekakwath balan...,96.078438
2,2,hey friendz මේ app 1n 1999 data package 1activ...,98.726120
3,3,"yako 3g wath hariyata deeganna baha, kohe 4g d...",96.047440
4,4,lebsack blaze mini 4g smartphone එක 3495 ta ga...,96.969696


In [ ]:
singlish_sentence = "oyage nama mokakda 🤣🤣🤣"
output = generate_sinhala(singlish_sentence, singlish_to_sinhala_dict, tokenizer)
print(output)

ඔයාගේ නම මොකක්ද 🤣🤣🤣


In [ ]:
from tqdm import tqdm

def add_generated_sinhala(
    df,
    text_col="Sentence",
    output_col="Sinhala_Generated"
):
    df = df.copy()
    outputs = []

    for sent in tqdm(df[text_col], desc="Generating Sinhala", total=len(df)):
        try:
            sinhala_out = generate_sinhala(
                sent,
                singlish_to_sinhala_dict,
                tokenizer
            )
        except Exception:
            sinhala_out = None

        outputs.append(sinhala_out)

    df[output_col] = outputs
    return df

In [ ]:
df_romanized_5000 = add_generated_sinhala(
    df_romanized_5000,
    text_col="Sentence",
    output_col="Sinhala_Generated"
)

Generating Sinhala:  99%|█████████▉| 4955/5000 [11:53:48<09:21, 12.47s/it]

In [ ]:
len(df_romanized_1000)

1000

In [ ]:
df_romanized_1000 = add_generated_sinhala(
    df_romanized_1000,
    text_col="Sentence",
    output_col="Sinhala_Generated"
)

Generating Sinhala: 100%|██████████| 1000/1000 [3:58:42<00:00, 14.32s/it]


In [ ]:
df_romanized_1000.to_csv('/content/drive/MyDrive/Singlish-Transliterator/df_romanized_1000_converted')

In [ ]:
df_romanized_1000.head(50)

,Unnamed: 0,Sentence,romanized_confidence,Sinhala_Generated
165000,188416,ma padikam puro puro enna goba wahansala bakan...,100.000000,මා පදිකම් පුරෝ පුරෝ එන්න ගොබ වහන්සල බකන් නිලාන...
165001,94107,nawa gilunath ban chuun,100.000000,නවා ගිලුනත් බණ චූන්
165002,227453,usaviyata ihalin gihin marapu pirisa 60 kata v...,100.000000,උසවියට ඉහළින් ගිහින් මරපු පිරිස 60 කට වෑඩී අහි...
165003,66899,proud to be a sri lankan😜😜😀😀,100.000000,ප්‍රොඋද් ටො බී අ සාරි ලන්කන්😜😜😀😀
165004,142806,hama gedarakama oya prashne amma kenek wela ma...,100.000000,හම ගෙදරකම ඔය ප්‍රශ්නේ අම්මා කෙනෙක් වෙලා මටත් ප...
165005,223105,salli gara ganna wadak dan pvt limited kiyala ...,100.000000,සල්ලි ගරා ගන්න වෑඩක් දාන ප්ව්ට් ලිමිටඩ් කියලා ...
165006,197004,hehe mage username ekai comment dana widihai c...,100.000000,හෙහෙ මගෙ උසෙර්නමෙ එකයි කමෙන්ට් දාන විදිහඉ කොපි...
165007,212623,hehe police ballanage neethiya kohomada niwata...,100.000000,හෙහෙ පොලිස් බල්ලනගෙ නීතිය කොහොමද නිවටයෝ වගේ ෆෝ...
165008,41072,kuda mole avlak wage,100.000000,කුඩ මොලේ අව්ලක් වගේ
165009,10199,oya saththuwaththa thiyena idamata shot eka de...,100.000000,ඔය සත්තුවත්ත තියෙන ඉඩමට ෂොට් එක දෙන්න හාදන්නෙ


### Translate to english

In [ ]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
df_romanized_1000 = pd.read_csv('/content/drive/MyDrive/Singlish-Transliterator/df_romanized_1000_converted')

Mounted at /content/drive


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
       

In [ ]:
SRC_LANG = "sin_LK"
TGT_LANG = "eng_Latn"

tokenizer.src_lang = SRC_LANG
forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)

In [ ]:
def translate_batch(texts, batch_size=32):
    translations = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos_token_id,
                max_length=128,
                num_beams=4
            )

        translated = tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True
        )
        translations.extend(translated)

    return translations

In [ ]:
df_romanized_1000.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'Sentence', 'romanized_confidence',
       'Sinhala_Generated'],
      dtype='object')

In [ ]:
df_romanized_1000["Sinhala_Generated"].apply(type).value_counts()

,count
Sinhala_Generated,
<class 'str'>,998
<class 'float'>,2


In [ ]:
float_rows = df_romanized_1000[
    df_romanized_1000["Sinhala_Generated"].apply(lambda x: isinstance(x, float))
]

float_rows

,Unnamed: 0.1,Unnamed: 0,Sentence,romanized_confidence,Sinhala_Generated
454,165454,623,ඒ විනාඩියේ මේක google කරලා බලහං. estimates for...,97.530869,NaN
519,165519,137407,mata ahanna thiyenne menna mekayi obata sathut...,100.000000,NaN


In [ ]:
df_romanized_1000["Sinhala_Generated"] = (
    df_romanized_1000["Sinhala_Generated"]
    .fillna(df_romanized_1000["Sentence"])
)

In [ ]:
df_romanized_1000["Sinhala_Generated"].apply(type).value_counts()

,count
Sinhala_Generated,
<class 'str'>,1000


In [ ]:
df_romanized_1000["english_text"] = translate_batch(df_romanized_1000["Sinhala_Generated"].tolist())

100%|██████████| 32/32 [06:21<00:00, 11.94s/it]


In [ ]:
df_romanized_1000.to_csv('/content/drive/MyDrive/Singlish-Transliterator/df_romanized_1000_converted_translated.csv')

### Classify the intents

In [ ]:
INTENTS = [
    "Complaint",
    "Suggestion",
    "Feedback",
    "Greeting",
    "Inquiry",
    "Other"
]

In [ ]:
FEW_SHOT_EXAMPLES = [
    {
        "text": "The app keeps crashing 😡",
        "intent": "Complaint",
        "confidence": 0.95
    },
    {
        "text": "You should add dark mode to the app",
        "intent": "Suggestion",
        "confidence": 0.94
    },
    {
        "text": "I think the new update is great!",
        "intent": "Feedback",
        "confidence": 0.92
    },
    {
        "text": "Good morning 😊",
        "intent": "Greeting",
        "confidence": 0.98
    },
    {
        "text": "What time does the store open?",
        "intent": "Inquiry",
        "confidence": 0.96
    },
    {
        "text": "I like turtles",
        "intent": "Other",
        "confidence": 0.50
    }
]

In [ ]:
def build_prompt(text):
    examples_text = "\n".join([
        f'Text: "{ex["text"]}"\nIntent: {ex["intent"]}\nConfidence: {ex["confidence"]}'
        for ex in FEW_SHOT_EXAMPLES
    ])

    prompt = """
      You are an intent classification system.

      Allowed intents: Complaint, Request, Feedback, Greeting, Informational, Other

      Text: "My order never arrived"

      Return ONLY a JSON object like: {"intent": <label>, "confidence": <0-1>}
      """

    return prompt

In [ ]:
def build_prompt(text):
    prompt = f"""
You are an intent classification system.

Allowed intents: {', '.join(INTENTS)}
Always pick exactly one of these intents.

Example:
Text: "The app keeps crashing 😡"
Intent: Complaint
Confidence: 0.95

Now classify the following text.
Return ONLY a JSON object like: {{"intent": <label>, "confidence": <0-1>}}

Text: "{text}"
"""
    return prompt

### Load intent classification model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-1.5B-Instruct")
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2-1.5B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
LABEL_MAP = {
    "Problem": "Complaint",
    "Complaints": "Complaint",
    "Issue": "Complaint",
    "Hello": "Greeting",
    "Hi": "Greeting",
    "Suggestion": "Feedback",
    "Info": "Informational",
}

In [ ]:
import json
import re

def parse_output(generated_text):
    try:
        matches = re.findall(r"\{.*?\}", generated_text, re.DOTALL)
        if matches:
            last_json = matches[-1]
            data = json.loads(last_json)
            intent = LABEL_MAP.get(data.get("intent", "Other"), data.get("intent", "Other"))
            confidence = float(data.get("confidence", 0.0))
            if intent not in INTENTS:
                intent = "Other"
            return intent, confidence
        else:
            return "Other", 0.0
    except:
        return "Other", 0.0

In [ ]:
def classify_text(text):
    prompt = build_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        temperature=0.0,
        eos_token_id=tokenizer.eos_token_id
    )

    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return parse_output(generated_text)

In [ ]:
def tokenize_batch(texts):
    prompts = [build_prompt(t) for t in texts]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)
    return inputs.to(model.device)

In [ ]:
def classify_texts_batch(texts, batch_size=16):
    results = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenize_batch(batch_texts)

        # Generate all outputs in one forward pass
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=120,
                do_sample=False,
                temperature=0.0,
                eos_token_id=tokenizer.eos_token_id
            )

        # Decode each sequence
        generated_texts = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        # Parse each JSON output
        batch_results = [parse_output(t) for t in generated_texts]
        results.extend(batch_results)
    return results


In [ ]:
parsed = classify_texts_batch(df_romanized_1000["english_text"].tolist(), batch_size=16)
df_romanized_1000["intent"] = [p[0] for p in parsed]
df_romanized_1000["confidence"] = [p[1] for p in parsed]

In [ ]:
print(df_romanized_1000)

     Unnamed: 0.1  Unnamed: 0  \
0          165000      188416   
1          165001       94107   
2          165002      227453   
3          165003       66899   
4          165004      142806   
..            ...         ...   
995        165995       11902   
996        165996      165892   
997        165997      180567   
998        165998       38037   
999        165999       39070   

                                              Sentence  romanized_confidence  \
0    ma padikam puro puro enna goba wahansala bakan...            100.000000   
1                              nawa gilunath ban chuun            100.000000   
2    usaviyata ihalin gihin marapu pirisa 60 kata v...            100.000000   
3                         proud to be a sri lankan😜😜😀😀            100.000000   
4    hama gedarakama oya prashne amma kenek wela ma...            100.000000   
..                                                 ...                   ...   
995  parana horu set ekama thama therila th

In [ ]:
df_romanized_1000.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'Sentence', 'romanized_confidence',
       'Sinhala_Generated', 'english_text', 'intent', 'confidence'],
      dtype='object')

In [ ]:
df_romanized_1000.drop(columns=["Unnamed: 0.1", "Unnamed: 0"], inplace=True)

In [ ]:
df_romanized_1000.to_csv('/content/drive/MyDrive/Singlish-Transliterator/df_romanized_1000_final')

### =========================================================================

In [ ]:
# # Define file paths
# input_file_path = '/content/drive/MyDrive/Singlish-Back-Transliterator/Test_Data/Original/Sinhala Test set 2.txt'
# output_directory = '/content/drive/MyDrive/Singlish-Back-Transliterator/Test_Data/Separated/Test_Set_2'  # Ensure this directory exists

# # Define the maximum number of sentence pairs per file
# max_pairs_per_file = 3000  # Adjust this as needed

# # Initialize counters
# pair_count = 0
# file_count = 1

# # Open the first output file
# output_file = open(f'{output_directory}/output_part2_{file_count}.txt', 'w', encoding='utf-8')

# with open(input_file_path, 'r', encoding='utf-8') as infile:
#     # Loop through each line, storing pairs
#     for line in infile:
#         # Strip any leading/trailing whitespace
#         line = line.strip()

#         # Check if line is not empty
#         if line:
#             # Write the line to the current output file
#             output_file.write(line + '\n')
#             pair_count += 1

#             # After writing a pair (two lines), add a blank line
#             if pair_count % 2 == 0:
#                 output_file.write('\n')  # Add a blank line between pairs

#                 # Check if we have reached the max pairs limit for the current file
#                 if pair_count // 2 >= max_pairs_per_file:
#                     output_file.close()
#                     file_count += 1
#                     pair_count = 0
#                     output_file = open(f'{output_directory}/output_part2_{file_count}.txt', 'w', encoding='utf-8')

# # Close the last output file
# output_file.close()

In [ ]:
# %%time

# def classify_line(line):
#     """Classify the line as Romanized, Sinhala, or blank."""
#     line = line.strip()

#     # Check if the line is blank
#     if not line:
#         return "Blank"

#     # Check if the line contains Sinhala characters (Unicode range 0D80–0DFF)
#     if any('\u0D80' <= char <= '\u0DFF' for char in line):
#         return "Sinhala"

#     # If it's not blank and doesn't contain Sinhala characters, it's Romanized
#     return "Romanized"

# # Open the input file and read the content
# with open('/content/drive/MyDrive/Singlish-Back-Transliterator/Test_Data/Original/Sinhala Test set 2.txt', 'r', encoding='utf-8') as file:
#     # Read the lines from the file
#     lines = file.readlines()

# # Open a new file to write the filtered pairs
# with open('/content/drive/MyDrive/Singlish-Back-Transliterator/Test_Data/Results/Test_Data_2/Test_Data_Result_2.txt', 'w', encoding='utf-8') as output_file:

#     # Variables to hold Romanized and Sinhala pairs
#     romanized_sentence = None
#     sinhala_sentence = None

#     # Loop through the lines and classify them
#     for i, line in enumerate(lines, start=1):
#         classification = classify_line(line)

#         if classification == "Romanized":
#             romanized_sentence = line.strip()  # Store the Romanized line
#         elif classification == "Sinhala":
#             sinhala_sentence = line.strip()  # Store the Sinhala line

#         # If both Romanized and Sinhala sentences are identified
#         if romanized_sentence and sinhala_sentence:
#             # Count the number of words in the Romanized sentence
#             word_count = len(romanized_sentence.split())

#             output_file.write(f"Romanized Input: {romanized_sentence}\n")
#             output_file.write(f"Expected Sinhala: {sinhala_sentence}\n")


#             time_start = time.time()
#             output = generate_sinhala(re.sub(r'\s+', ' ', romanized_sentence).strip(), singlish_to_sinhala_dict, tokenizer)
#             time_end = time.time()

#             time_taken = time_end - time_start

#             output_file.write(f"Generated Sinhala: {output}\n")
#             output_file.write(f"Time taken: {time_taken} seconds\n")

#             print(f"Romanized Input: {romanized_sentence}")
#             print(f"Expected Sinhala: {sinhala_sentence}")
#             print(f"Generated Sinhala: {output}\n")

#             output_file.write("\n")  # Write a blank line for readability

#             # Reset the variables after processing the pair
#             romanized_sentence = None
#             sinhala_sentence = None